# Agent에서의 Tool Use

이 노트북은 기존 `src/tools.py` 경로를 바꾸지 않으면서 tool calling 튜토리얼을 추가한다. 별도의 교육용 레이어를 통해 tool registry, tool selection, structured tool call 개념을 설명한다.

## 학습 목표

- agent가 왜 tool을 쓰는지 이해한다.
- tool registry의 역할을 배운다.
- tool selection과 structured tool call을 실습한다.
- calculator, data, search tool의 출력을 비교한다.


## 개념 설명

이 코스 전체에서 쓰는 것과 같은 environment check부터 시작한다. 여러 Jupyter kernel이나 원격 환경을 오가고 있다면 특히 도움이 된다.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(sys.executable)

이 setup 셀은 `src/tools_extended.py`의 additive tooling helper를 가져온다. 원래 `src/tools.py`는 그대로 두고, notebook에서는 tutorial 친화적인 registry API를 중심으로 본다.


In [ ]:
import pandas as pd


from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(PROJECT_ROOT)

from src.tools_extended import ToolCall, build_default_registry

pd.set_option('display.max_colwidth', 140)
registry = build_default_registry()


agent는 retrieval과 일반 reasoning만으로 부족할 때 tool을 쓴다. tool registry는 agent가 호출할 수 있는 기능 목록을 통제된 형태로 제공한다. structured tool call은 그 호출을 자유 텍스트 뒤에 숨기지 않고 명시적으로 드러낸다.

이 노트북에서 다루는 예시는 다음과 같다.

- `calculator`: 산술 계산
- `data_tool`: 작은 표 데이터 처리
- `search_tool`: 간단한 similarity 기반 문서 검색

## 구현


In [ ]:
registry.list_tools()


structured call이 중요한 이유는 intent와 execution을 분리해주기 때문이다. 아래 셀은 명시적인 tool call object를 만들고, 이를 registry를 통해 실행한다.


In [ ]:
structured_calls = [
    ToolCall('calculator', {'expression': '12 * 3 + 4'}),
    ToolCall('data_tool', {'rows': [{'team': 'A', 'score': 8}, {'team': 'B', 'score': 10}, {'team': 'A', 'score': 6}], 'column': 'score', 'operation': 'mean'}),
    ToolCall('search_tool', {'query': 'launch date', 'documents': ['The launch date is May 5, 2025.', 'The pilot begins in March.', 'Finance approved the budget.']}),
]
structured_results = [registry.call(tool_call).to_dict() for tool_call in structured_calls]
pd.DataFrame(structured_results)


tool selection은 tool execution과 별개의 문제다. 단순한 agent라 하더라도 먼저 어떤 tool이 relevant한지 판단하고, 그다음 필요한 tool에 대해서만 structured call을 구성해야 한다.


In [ ]:
selection_examples = [
    'Calculate the pilot duration in days.',
    'Find which document mentions the launch date.',
    'Count how many rows are in this dataset.',
]
pd.DataFrame(
    {
        'query': selection_examples,
        'selected_tools': [', '.join(registry.select_tools(query)) for query in selection_examples],
    }
)


## 실험

좋은 실험은 서로 다른 structured request를 registry에 넣고, 출력이 얼마나 안정적으로 나오는지 비교하는 것이다. 이 과정을 보면 free-form prompting만으로 tool을 다루는 것보다 typed input이 왜 디버깅에 유리한지 체감할 수 있다.


In [ ]:
experiment_calls = [
    ToolCall('calculator', {'expression': '25 - 7'}),
    ToolCall('data_tool', {'rows': [{'latency': 0.8}, {'latency': 1.2}, {'latency': 1.0}], 'column': 'latency', 'operation': 'mean'}),
    ToolCall('search_tool', {'query': 'pilot window', 'documents': ['Pilot window: March 10, 2025 to April 4, 2025.', 'The governance memo explains ownership.', 'The FAQ lists tool guidance.']}),
]
experiment_results = [registry.call(tool_call).to_dict() for tool_call in experiment_calls]
pd.DataFrame(experiment_results)


## 결과 해석

출력을 보면 tool 패턴이 세 가지로 나뉜다. arithmetic tool은 정확한 수치를, data tool은 구조화된 요약을, search tool은 랭킹된 후보 텍스트를 반환한다. 실제 agent에서는 어떤 질문에 어떤 tool을 써야 할지를 결정하는 selection policy가 핵심이 된다.


In [ ]:
analysis_frame = pd.DataFrame(
    [
        {'tool': 'calculator', 'best_for': 'precise arithmetic and simple derived values'},
        {'tool': 'data_tool', 'best_for': 'small structured datasets and aggregates'},
        {'tool': 'search_tool', 'best_for': 'finding relevant text before synthesis'},
    ]
)
analysis_frame


## 핵심 정리

- 이 실험을 통해 tool registry는 agent capability를 명시적이고 안전하게 유지하는 데 중요하다는 점을 확인했다.
- tool selection은 reasoning 문제이고, tool execution은 interface 문제다.
- structured tool call은 raw text 기반 tool 요청보다 테스트와 디버깅이 쉽다.
- 면접에서는 tool use를 설명할 때 **registry, selection, execution, structured output** 네 요소를 분리해 말하면 설득력이 높다.
